In [17]:
!pip install torch torchvision ipykernel jupyter

In [18]:
import torch
import Value from .index

SyntaxError: invalid syntax (838485314.py, line 2)

In [ ]:
x1 = torch.tensor([2.0], dtype=torch.double, requires_grad=True)
x2 = torch.tensor([0.0], dtype=torch.double, requires_grad=True)
w1 = torch.tensor([-3.0], dtype=torch.double, requires_grad=True)
w2 = torch.tensor([1.0], dtype=torch.double, requires_grad=True)
b = torch.tensor([6.8813735870195432], dtype=torch.double, requires_grad=True)

n = x1*w1 + x2*w2 + b
o = torch.tanh(n)

print(o.data.item())
o.backward()

print('-----------')
print('x2-grad', x2.grad.item())
print('x2-data', x2.item())
print('w2-grad', w2.grad.item())
print('w2-data', w2.item())
print('x1-grad', x1.grad.item())
print('x1-data', x1.item())
print('w1-grad', w1.grad.item())
print('w1-data', w1.item())

0.7071067811865476
-----------
x2-grad 0.4999999999999999
x2-data 0.0
w2-grad 0.0
w2-data 1.0
x1-grad -1.4999999999999996
x1-data 2.0
w1-grad 0.9999999999999998
w1-data -3.0


In [ ]:
o

tensor([0.7071], dtype=torch.float64, grad_fn=<TanhBackward0>)

In [26]:
from typing import Union
import math

ValueAutogradBackPropFixedLike = Union['Value_Autograd_BackpropFixed', int, float]


class Value_Autograd_BackpropFixed:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.prev = set(_children)
        self._backward = lambda: None
        self._op = _op
        self.label = label
        self.grad = 0

    def __repr__(self):
        return f'Value(data={self.data})'

    def __add__(self, second_operand: ValueAutogradBackPropFixedLike):
        if not isinstance(second_operand, Value_Autograd_BackpropFixed):
            second_operand = Value_Autograd_BackpropFixed(
                float(second_operand))

        output = Value_Autograd_BackpropFixed(
            self.data + second_operand.data, (self, second_operand), '+')

        def _backward():
            # change sign here
            self.grad += output.grad * 1
            # change sign here
            second_operand.grad += output.grad * 1
        output._backward = _backward

        return output

    def __sub__(self, second_operand):
        return self + (-second_operand)

    def __truediv__(self, second_operand):
        return self * second_operand ** -1

    def __pow__(self, other):
        assert isinstance(other, (int, float)
                          ), 'only supporting int and floats'
        out = Value_Autograd_BackpropFixed(
            self.data**other, (self,), f'**(other)')

        def _backward():
            self.grad += out.grad * other * self.data ** (other - 1)
        out._backward = _backward

        return out

    def __neg__(self):
        return self * -1

    def __mul__(self, second_operand: ValueAutogradBackPropFixedLike):
        if not isinstance(second_operand, Value_Autograd_BackpropFixed):
            second_operand = Value_Autograd_BackpropFixed(
                float(second_operand))

        output = Value_Autograd_BackpropFixed(
            self.data * second_operand.data, (self, second_operand), '*')

        def _backward():
            self.grad += output.grad * second_operand.data
            second_operand.grad += output.grad * self.data
        output._backward = _backward

        return output

    def __rmul__(self, second_operand):
        return self * second_operand

    def tanh(self):
        data = self.data
        tanh_data = (math.exp(2*data) - 1)/(math.exp(2*data) + 1)
        output = Value_Autograd_BackpropFixed(tanh_data, (self, ), 'tanh')

        def _backward():
            self.grad += output.grad * (1 - tanh_data ** 2)
        output._backward = _backward
        return output

    def exp(self):
        output = Value_Autograd_BackpropFixed(
            math.exp(self.data), (self,), 'exp')

        def _backward():
            self.grad += output.data * output.grad
        output._backward = _backward

        return output

    def backward(self):
        topo = []
        visited = set()

        def build_topo(node):
            if node not in visited:
                visited.add(node)
                for child in node.prev:
                    build_topo(child)
                topo.append(node)

        build_topo(self)
        self.grad = 1.0000000
        for node in reversed(topo):
            node._backward()

In [30]:
import random

class Neuron:
    
    def __init__(self, number_of_inputs):
        self.weight = [Value_Autograd_BackpropFixed(random.uniform(-1,1)) for _ in range(number_of_inputs)]
        self.b = Value_Autograd_BackpropFixed(random.uniform(-1,1))
        
    def __call__(self, x):
        act = sum((wi*xi for wi,xi in zip(self.weight, x)), self.b)
        out = act.tanh()
        return out
    
x = [2.0, 3.0]
n = Neuron(2)
n(x)

Value(data=0.778193579451303)